# Surgical Instrument Detection — YOLOv8m Training
**Dataset**: Surgical instrument detection (57 classes, 5637 images, Dutch hospital instruments)

**Run on**: Kaggle (free T4 GPU) or Google Colab

**Steps**: Upload dataset zip → auto train/val split → train YOLOv8m → export ONNX

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
!pip install -q ultralytics

In [ ]:
# ── 2. Upload & extract dataset ──────────────────────────────────────────
# Zip the dataset folder on your Mac first:
#   cd ~/Downloads
#   zip -r surgical_dataset.zip "Surgical instrument detection.v1i.yolov8"
# Then upload surgical_dataset.zip here.

import os, shutil

try:
    from google.colab import files
    uploaded = files.upload()  # upload surgical_dataset.zip
    zip_name = list(uploaded.keys())[0]
    !unzip -q "{zip_name}" -d /content/dataset
    DATASET_ROOT = "/content/dataset/Surgical instrument detection.v1i.yolov8"
except ImportError:
    # Kaggle: upload zip via 'Add Data' panel or place in /kaggle/working/
    !unzip -q /kaggle/working/surgical_dataset.zip -d /kaggle/working/dataset
    DATASET_ROOT = "/kaggle/working/dataset/Surgical instrument detection.v1i.yolov8"

print("Dataset root:", DATASET_ROOT)
print("Contents:", os.listdir(DATASET_ROOT))

In [ ]:
# ── 3. Create train/val split (80/20) ───────────────────────────────────
# Dataset only has 'train/' — we split it into train/val
import random
from pathlib import Path

random.seed(42)

src_images = Path(DATASET_ROOT) / "train" / "images"
src_labels = Path(DATASET_ROOT) / "train" / "labels"

train_img = Path(DATASET_ROOT) / "images" / "train"
val_img   = Path(DATASET_ROOT) / "images" / "val"
train_lbl = Path(DATASET_ROOT) / "labels" / "train"
val_lbl   = Path(DATASET_ROOT) / "labels" / "val"

for p in [train_img, val_img, train_lbl, val_lbl]:
    p.mkdir(parents=True, exist_ok=True)

all_images = list(src_images.glob("*.jpg"))
random.shuffle(all_images)

val_size  = int(len(all_images) * 0.2)
val_files = all_images[:val_size]
trn_files = all_images[val_size:]

def copy_split(files, img_dst, lbl_dst):
    for img in files:
        shutil.copy(img, img_dst / img.name)
        lbl = src_labels / img.with_suffix(".txt").name
        if lbl.exists():
            shutil.copy(lbl, lbl_dst / lbl.name)

copy_split(trn_files, train_img, train_lbl)
copy_split(val_files,  val_img,  val_lbl)

print(f"Train: {len(trn_files)} images")
print(f"Val:   {len(val_files)} images")

In [ ]:
# ── 4. Write data.yaml ───────────────────────────────────────────────────
import yaml

CLASS_NAMES = [
    'Algemeen_Aspiratiecanule_CH_D3-3_mm_WL11_cm', 'Algemeen_Cuvette_Bot_3mm',
    'Algemeen_Cuvette_Bot_5mm', 'Algemeen_Cuvette_Bot_8mm',
    'Algemeen_Deperiosteur_Rugine_Geb_6mm_18cm', 'Algemeen_Doekklem_Scherp_15cm',
    'Algemeen_Doekklem_Stomp_Bol_10-12cm', 'Algemeen_Drevel_5mm',
    'Algemeen_Hamer_300g', 'Algemeen_Heft_3', 'Algemeen_Heft_4',
    'Algemeen_Klem_A_Crochet_14cm', 'Algemeen_Klem_A_Crochet_20_cm',
    'Algemeen_Klem_A_Leriche_Geb_15cm', 'Algemeen_Klem_A_Mosquito_Geb_12cm',
    'Algemeen_Klem_C_Kocher_18cm', 'Algemeen_Klem_C_Mosquito_12cm',
    'Algemeen_Klem_C_Mosquito_Geb_12cm', 'Algemeen_Knabbeltang_2-7_3_mm_14_cm',
    'Algemeen_Knabbeltang_Geb_Blumenthal_12-16_cm', 'Algemeen_Knabbeltang_Ruskin_Geb_19_cm',
    'Algemeen_Kniptang_Bot_Ruskin-Liston_30deg_18cm', 'Algemeen_Pincet_A_Debakey_2-7_mm_20cm',
    'Algemeen_Recipient', 'Algemeen_Redonnaald_CH_14', 'Algemeen_Redonnaald_Ch_9_10',
    'Algemeen_Schaar_Jameson_Geb_15cm', 'Algemeen_Schaar_Mayo_17cm',
    'Algemeen_Schaar_Mayo_Geb_17cm', 'Algemeen_Spreidde_Weitlaner_2_3T_Scherp_12cm',
    'Algemeen_Spreidde_Weitlaner_3_4T_stomp_13cm', 'Algemeen_Spreidde_Weitlaner_3_4_T_Scherp_16cm',
    'Algemeen_Wondhaak_Farabeuf_15cm', 'Algemeen_Wondhaak_Gillies_19cm',
    'Algemeen_Wondhaak_Hofmann_6_mm_16cm', 'Algemeen_Wondhaak_Howarth-Morel_Scherp_Stomp_22cm',
    'Algemeen_Wondhaak_Langenbeck_16_30mm', 'Algemeen_Wondhaak_Langenbeck_6_25_mm',
    'Algemeen_Wondhaak_Ragnell_14cm', 'Algemeen_Wondhaak_Ragnell_16cm',
    'Algemeen_Wondhaak_Senn_Miller_15cm', 'Algemeen_Wondhaak_Volkmann_Scherp_6T',
    'Algemeen_deperiosteur_Rugine_Geb_10_mm_21_cm', 'Algemeen_kniptang_Metaal_22_23_cm',
    'Algemeen_pincet_A_15cm', 'Algemeen_pincet_A_Adson_12cm',
    'Algemeen_pincet_C_15cm', 'Algemeen_pincet_C_Adson_12cm',
    'Algemeen_pincet_C_Gillies_15_cm', 'Algemeen_plooitang_plat_16cm',
    'Algemene_Klem_Reductie_13cm', 'Algemene_Naaldvoerder_Hegar_TC_18cm',
    'Algemene_Naaldvoerder_Mathieu_TC_17cm', 'Algemene_Naaldvoerder_crile-wood_TC_15cm',
    'Algemene_Schaar_Metzenbaum_Geb_18cm', 'Algemene_klem_A_Mosquito_12cm', 'FijneDarm'
]

data_cfg = {
    "path": DATASET_ROOT,
    "train": "images/train",
    "val": "images/val",
    "nc": 57,
    "names": CLASS_NAMES,
}

DATA_YAML = str(Path(DATASET_ROOT) / "data_split.yaml")
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, allow_unicode=True)

print("data.yaml written:", DATA_YAML)

In [ ]:
# ── 5. Train YOLOv8m ─────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=10.0,
    flipud=0.5,
    project="surgical_training",
    name="yolov8m_surgical",
    device=0,
    workers=4,
)

best_pt = str(Path(results.save_dir) / "weights" / "best.pt")
print("Best model:", best_pt)

In [ ]:
# ── 6. Validate ──────────────────────────────────────────────────────────
model_best = YOLO(best_pt)
metrics = model_best.val(data=DATA_YAML, imgsz=640)
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# ── 7. Export to ONNX ────────────────────────────────────────────────────
model_best.export(
    format="onnx",
    imgsz=640,
    opset=11,
    simplify=True,
    dynamic=False,
)
onnx_path = best_pt.replace(".pt", ".onnx")
print("ONNX saved:", onnx_path)

In [ ]:
# ── 8. Download artifacts ────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(best_pt)
    files.download(onnx_path)
    print("Downloaded best.pt and best.onnx")
except ImportError:
    import shutil
    shutil.copy(best_pt,  "/kaggle/working/best.pt")
    shutil.copy(onnx_path, "/kaggle/working/best.onnx")
    print("Saved to /kaggle/working/ — download from Kaggle output panel")